# 🛠️ EMIPredict AI: Preprocessing Deep-Dive

This notebook breaks down the `preprocessing.py` script into interactive steps. Preprocessing is the most critical stage of the Machine Learning pipeline, where raw data is transformed into a format that models can understand.

## 1. Environment Setup
We need basic data manipulation libraries (`pandas`, `numpy`) and preprocessing tools from `sklearn`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import pickle
import os
from pathlib import Path

# Set project root path
project_root = Path('..')
data_path = project_root / 'data/cleaned/cleaned_emi_prediction_dataset.parquet'

print(f"Project Root: {project_root.get_absolute()}")

## 2. Load the Cleaned Dataset
We start with the data that has already been cleaned (missing values handled, types corrected).

In [ ]:
df = pd.read_parquet(data_path)
print(f"Initial Dataset Shape: {df.shape}")
df.head()

## 3. Feature Engineering: Creating "Domain Knowledge" Columns
Raw data like 'Monthly Salary' or 'Rent' is good, but models perform better if we give them **Ratios** and **Aggregates**.

In [ ]:
# 1. Total monthly expenses: Sum of all recurring costs
expense_cols = ['monthly_rent', 'school_fees', 'college_fees',
                'travel_expenses', 'groceries_utilities', 
                'other_monthly_expenses', 'current_emi_amount']
df['total_monthly_expenses'] = df[expense_cols].fillna(0).sum(axis=1)

# 2. Disposable income: What's left after all expenses?
df['disposable_income'] = df['monthly_salary'] - df['total_monthly_expenses']

# 3. EMI-to-income ratio: The proportion of salary the user spends on the requested EMI
df['emi_to_income_ratio'] = df['requested_amount'] / (df['monthly_salary'] * df['requested_tenure'])

# 4. Savings ratio: Liquidity vs. monthly income
df['savings_ratio'] = (df['bank_balance'] + df['emergency_fund'].fillna(0)) / (df['monthly_salary'] + 1)

# 5. Debt burden: Current debt load vs. income
df['debt_burden'] = df['current_emi_amount'].fillna(0) / (df['monthly_salary'] + 1)

# 6. Family financial pressure: How much burden per dependent?
df['family_pressure'] = df['dependents'] / (df['monthly_salary'] / 10000 + 1)

print(f"Total columns after engineering: {df.shape[1]}")

## 4. Categorical Encoding
Machine Learning models only understand numbers. We convert categorical text (e.g., 'Male'/'Female') into integers using `LabelEncoder`.

In [ ]:
cat_cols = ['gender', 'marital_status', 'education', 'employment_type',
            'company_type', 'house_type', 'emi_scenario', 'existing_loans']

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f"Encoded {col}: {le.classes_}")

## 5. Target Preparation
We have two targets:
1. `emi_eligibility` (Classification - Approved/Rejected)
2. `max_monthly_emi` (Regression - Amount)

In [ ]:
target_clf = 'emi_eligibility'
target_reg = 'max_monthly_emi'

le_target = LabelEncoder()
df[target_clf] = le_target.fit_transform(df[target_clf].astype(str))
encoders['emi_eligibility'] = le_target

print("Target classes index map:", {i: c for i, c in enumerate(le_target.classes_)})

## 6. Train/Test Split
We reserve 20% of the data to test how the model performs on unseen examples.

In [ ]:
X = df.drop(columns=[target_clf, target_reg])
yc = df[target_clf]
yr = df[target_reg]

X_train, X_test, yc_train, yc_test, yr_train, yr_test = train_test_split(
    X, yc, yr, test_size=0.2, random_state=42, stratify=yc
)

print(f"Training Set size: {X_train.shape}")
print(f"Testing Set size: {X_test.shape}")

## 7. Feature Scaling
Features have different ranges (Age: 20-60, Salary: 20,000-200,000). To prevent large numbers from dominating the model, we use `StandardScaler` to bring all values to a mean of 0 and standard deviation of 1.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head()

## 8. Persistence (Saving for Production)
We must save the `scaler` and `encoders` so that when a user enters data in our Streamlit app, we can apply the **exact same** transformation.

In [ ]:
models_dir = project_root / 'models'
models_dir.mkdir(exist_ok=True)

pickle.dump(scaler, open(models_dir / 'scaler.pkl', 'wb'))
pickle.dump(encoders, open(models_dir / 'encoders.pkl', 'wb'))

print(f"✅ Scaler and encoders saved to {models_dir.absolute()}")